# trying to run svg model

In [3]:
import sys

sys.path.append('SVG/')
sys.path.append('SVG/autoencoder')

In [4]:
import os
import glob
import torch
from torchvision.utils import save_image
from PIL import Image
from IPython.display import display
from omegaconf import OmegaConf

from rectified_flow.rectified_flow import RectifiedFlow
from utils import instantiate_from_config
from utils import find_model
from ldm.models.dino_decoder import DinoDecoder



In [10]:

# -------------------------
# Global setup
# -------------------------
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"


# -------------------------
# 1. Load config
# -------------------------
def get_config(ckpt_path):
    """Automatically locate the corresponding config for a checkpoint."""
    exp_root = "/".join(ckpt_path.split("/")[:-1])
    print(exp_root)
    config_path = glob.glob(os.path.join(exp_root, "*.yaml"))
    assert len(config_path) == 1, f"Expected one config file under {exp_root}"
    config = OmegaConf.load(config_path[0])
    exp_name = os.path.basename(exp_root)
    return exp_name, config


# === User-defined path ===

ckpt_path = "data/V1-SVG-XL-7000K-256x256.pt"  # TODO: fill with checkpoint path
assert ckpt_path and os.path.exists(ckpt_path), "Please set a valid ckpt_path"
exp_name, config = get_config(ckpt_path)
step = os.path.splitext(os.path.basename(ckpt_path))[0]
print(f"Experiment: {exp_name}")



data
Experiment: data


In [14]:

# -------------------------
# 2. Load main model & decoder
# -------------------------
# Load DiT backbone
model = instantiate_from_config(config.model)
state_dict = find_model(ckpt_path)
model.load_state_dict(state_dict, strict=False)
model = model.to(device).eval()


Model loading time: 0.10 seconds
load ema ckpt


In [17]:
config

{'basic': {'exp_name': 'SVG-XL', 'results_dir': 'exps', 'data_path': None, 'global_seed': 1234, 'epochs': 1000, 'log_every': 100, 'ckpt_every': 50000, 'rf': True, 'shift': 0.4, 'accum_iter': 1, 'clip_grad_norm': None, 'image_size': 256, 'global_batch_size': 256, 'num_workers': 16, 'timestep_start': 0, 'timestep_end': 1000, 'encoder': 'dinov3_vitsp16', 'feature_norm': True, 'encoder_config': 'visualizations/svg_autoencoder.yaml'}, 'model': {'ckpt': None, 'target': 'models.models_SVG.DiT', 'params': {'input_size': 16, 'num_classes': 1000, 'patch_size': 1, 'depth': 28, 'hidden_size': 1152, 'num_heads': 16, 'mlp_ratio': 4, 'use_swiglu': False, 'in_channels': 392, 'qk_norm': True, 'class_dropout_prob': 0.1}}, 'optim': {'base_learning_rate': 0.0001, 'weight_decay': 0, 'betas': [0.9, 0.999]}, 'lr_sheduler': {'warmup': None, 'train_epoch': None}}

In [21]:
from visualizations.extract_svg_features import load_dinov3_encoder
dinov3 = load_dinov3_encoder(
    'data/dinov3_vits16plus_pretrain_lvd1689m.pth',
    'dinov3'
)
# Load DinoDecoder from encoder config

# encoder_config = OmegaConf.load(config.basic.encoder_config)
# dinov3 = instantiate_from_config(encoder_config.model).eval()
# z_channels = encoder_config.model.params.ddconfig.z_channels


Loading DINOv3 from checkpoint: data/dinov3_vits16plus_pretrain_lvd1689m.pth
Loading checkpoint...
Loading DINOv3 architecture from local repository: dinov3
DINOv3 architecture loaded from local repository
✓ DINOv3 weights loaded successfully!


In [22]:
z_channels = 392

In [23]:
dinov3

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): SwiGLUFFN(
        (w1): Linear(in_features=384, out_features=1536, bias=True)
        (w2): Linear(in_features=384, out_features=1536, bias=True)
        (w3): Linear(in_features=1536, out_features=384, bias=True)
      )
      (ls2): LayerScale()
    )
  )
  (norm): LayerNorm((384,), ep

In [24]:


# -------------------------
# 3. Sampling setup
# -------------------------
seed = 0
torch.manual_seed(seed)

num_steps = 25
cfg_scale = 4
image_size = 256
samples_per_row = 4

# Example class labels (ImageNet indices)
class_labels = [15, 270, 284, 688, 250, 146, 980, 484, 207, 20, 387, 974, 88, 979, 417, 279]
n = len(class_labels)

# Initialize latent z
z = torch.randn(n, (image_size // 16) ** 2, z_channels, device=device)

# Load feature normalization stats if applicable
# stats_path = "dinov3_sp_stats.pt"
# assert os.path.exists(stats_path), "Missing dinov3_sp_stats.pt"
# stats = torch.load(stats_path)
# sp_mean = stats["dinov3_sp_mean"].to(device)[:, :, :z_channels]
# sp_std = stats["dinov3_sp_std"].to(device)[:, :, :z_channels]

# CFG and timestep settings
timestep_shift = 0.15
cfg_mode = "constant"
mode = "euler"

y = torch.tensor(class_labels, device=device)
y_null = torch.full_like(y, 1000)


In [25]:

# Run sampling
diffusion = RectifiedFlow(model)
print(f"Sampling with cfg_mode={cfg_mode}, steps={num_steps}, cfg={cfg_scale}")

samples = diffusion.sample(
    z,
    cond=y,
    null_cond=y_null,
    sample_steps=num_steps,
    cfg=cfg_scale,
    mode=mode,
    timestep_shift=timestep_shift,
    cfg_mode=cfg_mode,
)

# Apply feature normalization if needed
if config.basic.get("feature_norm", False):
    samples = samples #* sp_std + sp_mean

# Reshape [B, T, D] -> [B, D, H, W]
B, T, D = samples.shape
samples_latent = samples.permute(0, 2, 1).reshape(B, D, image_size // 16, image_size // 16)



Sampling with cfg_mode=constant, steps=25, cfg=4


KeyboardInterrupt: 

In [ ]:

# -------------------------
# 4. Decode to full-resolution images
# -------------------------
with torch.no_grad():
    decoded_full = dinov3.decode(samples_latent)
print(decoded_full.shape)

decoded_full = torch.clamp(decoded_full, -1, 1)

# Save and visualize
save_path = (
    f"{cfg_mode}_sample_{exp_name}_{step}_"
    f"steps{num_steps}_{mode}_cfg{cfg_scale}_shift{timestep_shift}_{image_size}.png"
)
save_image(decoded_full, save_path, nrow=samples_per_row, normalize=True, value_range=(-1, 1))
display(Image.open(save_path))

print(f"Saved results to {save_path}")
